#Mount Drive + go to repo root

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant"
!ls -la

Mounted at /content/drive
/content/drive/My Drive/Agentic AI Projects/adjuster_assistant
total 17
drwx------ 2 root root 4096 Feb 20 01:25 app
drwx------ 2 root root 4096 Feb 20 01:25 data
-rw------- 1 root root  183 Feb 20 21:21 .env
drwx------ 2 root root 4096 Feb 20 01:25 notebooks
drwx------ 2 root root 4096 Feb 20 01:25 tests


In [ ]:
!find "data" -maxdepth 3 -type f | sed -e 's|^| - |'

 - data/sample/policies/END_WATER_001_POL123_TX_v1.pdf.gdoc
 - data/sample/policies/END_WIND_001_POL123_TX_v1.pdf.gdoc
 - data/sample/policies/Dec Metadata.json.gdoc
 - data/sample/policies/Base.json.gdoc
 - data/sample/policies/Endorsements.json.gdoc
 - data/sample/policies/DEC_CP_POL123_TX_v1.pdf
 - data/sample/policies/DEC_CP_POL123_TX_v2.pdf.gdoc
 - data/sample/policies/CP00_10_BaseForm_POL123_TX_v1.pdf
 - data/sample/policies/CP00_10_BaseForm_POL123_TX_v2.pdf.gdoc
 - data/sample/claims/CLM1001_FNOL_WaterLeak_TX_v1.pdf.gdoc
 - data/sample/claims/CLM1001_AdjusterNotes_TX_v1.pdf.gdoc
 - data/sample/claims/CLM1001_Estimate_And_Mitigation_TX_v1.pdf.gdoc
 - data/sample/claims/FNOL.json.gdoc
 - data/sample/claims/Adjuster Notes.json.gdoc
 - data/sample/claims/EstimationAndMitigation.json.gdoc
 - data/chroma_db/chroma.sqlite3


#Install Agents SDK

In [ ]:
!pip -q install openai chromadb pypdf python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
from dotenv import load_dotenv
import os

loaded = load_dotenv("/content/drive/MyDrive/Agentic AI Projects/agentic_adjuster_assistant/.env")
print("dotenv loaded:", loaded)
print("key present:", bool(os.getenv("OPENAI_API_KEY")))

dotenv loaded: True
key present: True


In [ ]:
#Patch for vectorstore.py
%%bash
set -e
cd "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant"

cat > app/rag/vectorstore.py <<'PY'
from __future__ import annotations
from typing import List, Dict, Any, Optional
import chromadb
from chromadb.config import Settings

def _normalize_where(where: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Chroma requires a single top-level operator in `where`.
    If caller passes multiple field filters, wrap them in $and.
    """
    if not where:
        return {}
    if len(where) == 1:
        return where
    return {"$and": [{k: v} for k, v in where.items()]}

class ChromaStore:
    def __init__(self, persist_dir: str, collection_name: str = "agentic_adjuster_assistant"):
        self.client = chromadb.PersistentClient(
            path=persist_dir,
            settings=Settings(anonymized_telemetry=False),
        )
        self.col = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

    def upsert(self, ids: List[str], embeddings: List[List[float]], documents: List[str], metadatas: List[Dict[str, Any]]):
        self.col.upsert(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)

    def query(self, query_embedding: List[float], where: Optional[Dict[str, Any]] = None, top_k: int = 8):
        return self.col.query(query_embeddings=[query_embedding], n_results=top_k, where=_normalize_where(where))

    def delete_by_filter(self, where: Dict[str, Any]):
        self.col.delete(where=_normalize_where(where))
PY

echo "✅ Patched app/rag/vectorstore.py"

✅ Patched app/rag/vectorstore.py


In [ ]:
%%bash
set -e
cd "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant/app/rag"

# Create the app/rag directory if it doesn't exist
mkdir -p "app/rag"

# ---------------- loaders.py ----------------
cat > loaders.py <<'PY'
from __future__ import annotations
from dataclasses import dataclass
from typing import List
from pypdf import PdfReader

@dataclass
class PageText:
    page: int
    text: str

def load_pdf_pages(pdf_path: str) -> List[PageText]:
    reader = PdfReader(pdf_path)
    pages: List[PageText] = []
    for i, page in enumerate(reader.pages, start=1):
        txt = page.extract_text() or ""
        txt = txt.replace("\u00a0", " ").strip()
        pages.append(PageText(page=i, text=txt))
    return pages
PY

# ---------------- vectorstore.py ----------------
cat > app/rag/vectorstore.py <<'PY'
from __future__ import annotations
from typing import List, Dict, Any, Optional
import chromadb
from chromadb.config import Settings

def _normalize_where(where: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Chroma requires a single top-level operator in `where`.
    If caller passes multiple field filters, wrap them in $and.
    """
    if not where:
        return {}
    if len(where) == 1:
        return where
    return {"$and": [{k: v} for k, v in where.items()]}

class ChromaStore:
    def __init__(self, persist_dir: str, collection_name: str = "agentic_adjuster_assistant"):
        self.client = chromadb.PersistentClient(
            path=persist_dir,
            settings=Settings(anonymized_telemetry=False),
        )
        self.col = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

    def upsert(self, ids: List[str], embeddings: List[List[float]], documents: List[str], metadatas: List[Dict[str, Any]]):
        self.col.upsert(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)

    def query(self, query_embedding: List[float], where: Optional[Dict[str, Any]] = None, top_k: int = 8):
        return self.col.query(query_embeddings=[query_embedding], n_results=top_k, where=_normalize_where(where))

    def delete_by_filter(self, where: Dict[str, Any]):
        self.col.delete(where=_normalize_where(where))
PY

# ---------------- ingest_common.py ----------------
cat > ingest_common.py <<'PY'
from __future__ import annotations
from typing import Callable, Dict, Any, List, Tuple, Protocol, Optional
from openai import OpenAI
from loaders import load_pdf_pages
from vectorstore import ChromaStore

class ChunkLike(Protocol):
    chunk_id: str
    text: str
    metadata: Dict[str, Any]

ChunkerFn = Callable[[List[Tuple[int, str]], Dict[str, Any], int, int], List[ChunkLike]]

def ingest_pdf(
    *,
    pdf_path: str,
    base_metadata: Dict[str, Any],
    chunker: ChunkerFn,
    persist_dir: str,
    collection_name: str = "agentic_adjuster_assistant",
    embed_model: str = "text-embedding-3-small",
    max_chars: int = 1600,
    overlap: int = 200,
    reingest_filter: Optional[Dict[str, Any]] = None,
) -> int:
    pages = load_pdf_pages(pdf_path)
    page_tuples = [(p.page, p.text) for p in pages]
    chunks = chunker(page_tuples, base_metadata, max_chars, overlap)
    if not chunks:
        return 0

    store = ChromaStore(persist_dir=persist_dir, collection_name=collection_name)
    if reingest_filter:
        store.delete_by_filter(reingest_filter)

    client = OpenAI()
    texts = [c.text for c in chunks]
    resp = client.embeddings.create(model=embed_model, input=texts)
    vectors = [d.embedding for d in resp.data]

    store.upsert(
        ids=[c.chunk_id for c in chunks],
        embeddings=vectors,
        documents=[c.text for c in chunks],
        metadatas=[c.metadata for c in chunks],
    )
    return len(chunks)
PY

# ---------------- ingest_policy.py ----------------
cat > ingest_policy.py <<'PY'
from __future__ import annotations
from typing import Dict, Any, List, Tuple
from policy_chunking import chunk_policy_pages
from ingest_common import ingest_pdf

def _policy_chunker(pages: List[Tuple[int, str]], base_md: Dict[str, Any], max_chars: int, overlap: int):
    return chunk_policy_pages(pages, base_metadata=base_md, max_chars=max_chars, overlap=overlap)

def ingest_policy_pdf(
    *,
    pdf_path: str,
    policy_metadata: Dict[str, Any],
    persist_dir: str,
    collection_name: str = "agentic_adjuster_assistant",
    embed_model: str = "text-embedding-3-small",
    max_chars: int = 1600,
    overlap: int = 200,
    reingest: bool = True,
) -> int:
    reingest_filter = None
    if reingest and policy_metadata.get("doc_id") and policy_metadata.get("doc_version"):
        reingest_filter = {"doc_id": policy_metadata["doc_id"], "doc_version": policy_metadata["doc_version"]}
    return ingest_pdf(
        pdf_path=pdf_path,
        base_metadata=policy_metadata,
        chunker=_policy_chunker,
        persist_dir=persist_dir,
        collection_name=collection_name,
        embed_model=embed_model,
        max_chars=max_chars,
        overlap=overlap,
        reingest_filter=reingest_filter,
    )
PY

# ---------------- ingest_claims.py ----------------
cat > ingest_claims.py <<'PY'
from __future__ import annotations
from typing import Dict, Any, List, Tuple
from claims_chunking import chunk_claim_document
from ingest_common import ingest_pdf

def _claims_chunker(pages: List[Tuple[int, str]], base_md: Dict[str, Any], max_chars: int, overlap: int):
    return chunk_claim_document(pages=pages, base_metadata=base_md, max_chars=max_chars, overlap=overlap)

def ingest_claim_pdf(
    *,
    pdf_path: str,
    claim_metadata: Dict[str, Any],
    persist_dir: str,
    collection_name: str = "agentic_adjuster_assistant",
    embed_model: str = "text-embedding-3-small",
    max_chars: int = 1800,
    overlap: int = 200,
    reingest: bool = True,
) -> int:
    required = ["claim_id", "policy_id", "doc_type", "doc_id", "doc_version"]
    missing = [k for k in required if not claim_metadata.get(k)]
    if missing:
        raise ValueError(f"Missing required claim_metadata keys: {missing}")
    reingest_filter = {"doc_id": claim_metadata["doc_id"], "doc_version": claim_metadata["doc_version"]} if reingest else None
    return ingest_pdf(
        pdf_path=pdf_path,
        base_metadata=claim_metadata,
        chunker=_claims_chunker,
        persist_dir=persist_dir,
        collection_name=collection_name,
        embed_model=embed_model,
        max_chars=max_chars,
        overlap=overlap,
        reingest_filter=reingest_filter,
    )
PY

echo "✅ Created app/rag ingestion modules"

✅ Created app/rag ingestion modules


In [ ]:
#Policy Chunking

%%bash
set -e
cd "/content/drive/MyDrive/Agentic AI Projects/agentic_adjuster_assistant/app/rag"

# ---------------- policy chunking.py ----------------
cat > policy_chunking.py <<'PY'
from __future__ import annotations
import re
from dataclasses import dataclass
from typing import List, Dict, Any, Iterable, Tuple

@dataclass
class Chunk:
    chunk_id: str
    text: str
    metadata: Dict[str, Any]

_HEADING_RE = re.compile(r"^(?:(\d+(\.\d+)*)\s+)?([A-Z][A-Z0-9 \-/:]{6,}|[A-Z][a-z][\w \-/:]{4,})\s*$")

def _split_by_headings(text: str) -> List[Tuple[str, str]]:
    lines = [ln.strip() for ln in text.splitlines()]
    blocks: List[Tuple[str, List[str]]] = []
    current_heading = "BODY"
    current: List[str] = []
    found = False
    for ln in lines:
        if not ln:
            continue
        if _HEADING_RE.match(ln) and len(ln) <= 90:
            found = True
            if current:
                blocks.append((current_heading, current))
            current_heading = ln
            current = []
        else:
            current.append(ln)
    if current:
        blocks.append((current_heading, current))
    if not found:
        return [("BODY", text)]
    return [(h, "\n".join(b).strip()) for h, b in blocks if b]

def _window_split(text: str, max_chars: int = 1600, overlap: int = 200) -> List[str]:
    text = text.strip()
    if len(text) <= max_chars:
        return [text]
    out = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        out.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [x for x in out if x]

def chunk_policy_pages(
    pages: Iterable[Tuple[int, str]],
    base_metadata: Dict[str, Any],
    max_chars: int = 1600,
    overlap: int = 200,
) -> List[Chunk]:
    chunks: List[Chunk] = []
    idx = 0
    for page_num, page_text in pages:
        if not page_text.strip():
            continue
        blocks = _split_by_headings(page_text)
        for heading, body in blocks:
            parts = _window_split(body, max_chars=max_chars, overlap=overlap)
            for p in parts:
                idx += 1
                chunk_id = f"pol_{base_metadata.get('policy_id','unknown')}_p{page_num}_{idx}"
                md = dict(base_metadata)
                md.update({"page": page_num, "section": heading, "doc_type": "policy", "chunk_id": chunk_id})
                chunks.append(Chunk(chunk_id=chunk_id, text=p, metadata=md))
    return chunks
PY

In [ ]:
#claims_chunking.py

%%bash
set -e
cd "/content/drive/MyDrive/Agentic AI Projects/agentic_adjuster_assistant/app/rag"

# ---------------- claims chunking.py ----------------
cat > claims_chunking.py <<'PY'
from __future__ import annotations
import re
from dataclasses import dataclass
from typing import Dict, Any, List, Iterable, Tuple

@dataclass
class Chunk:
    chunk_id: str
    text: str
    metadata: Dict[str, Any]

_HEADING_RE = re.compile(r"^(?:(\d+(\.\d+)*)\s+)?([A-Z][A-Z0-9 \-/:]{6,}|[A-Z][a-z][\w \-/:]{4,})\s*$")
_NOTE_TS_RE = re.compile(
    r"^(\d{4}-\d{2}-\d{2}|\d{1,2}/\d{1,2}/\d{2,4})"
    r"(\s+\d{1,2}:\d{2}(\s*[AP]M)?)?"
    r"(\s*[-–—]\s*)?.+",
    re.IGNORECASE
)

def _window_split(text: str, max_chars: int, overlap: int) -> List[str]:
    text = text.strip()
    if len(text) <= max_chars:
        return [text]
    out: List[str] = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        out.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [x for x in out if x]

def _split_by_headings(text: str) -> List[Tuple[str, str]]:
    lines = [ln.strip() for ln in text.splitlines()]
    blocks: List[Tuple[str, List[str]]] = []
    current_heading = "BODY"
    current: List[str] = []
    found = False
    for ln in lines:
        if not ln:
            continue
        if _HEADING_RE.match(ln) and len(ln) <= 90:
            found = True
            if current:
                blocks.append((current_heading, current))
            current_heading = ln
            current = []
        else:
            current.append(ln)
    if current:
        blocks.append((current_heading, current))
    if not found:
        return [("BODY", text)]
    return [(h, "\n".join(b).strip()) for h, b in blocks if b]

def _split_claim_notes(text: str) -> List[str]:
    lines = text.splitlines()
    entries: List[List[str]] = []
    current: List[str] = []
    found = False
    for ln in lines:
        if _NOTE_TS_RE.match(ln.strip()):
            found = True
            if current:
                entries.append(current)
            current = [ln.strip()]
        else:
            if ln.strip():
                current.append(ln.strip())
    if current:
        entries.append(current)
    if not found:
        return [text.strip()]
    return ["\n".join(e).strip() for e in entries if e]

def chunk_claim_document(
    pages: Iterable[Tuple[int, str], Any],
    base_metadata: Dict[str, Any],
    max_chars: int = 1800,
    overlap: int = 200,
):
    doc_type = (base_metadata.get("doc_type") or "claim_doc").lower()
    claim_id = base_metadata.get("claim_id", "unknown")
    doc_id = base_metadata.get("doc_id", "doc_unknown")

    chunks: List[Chunk] = []
    idx = 0
    for page_num, page_text in pages:
        if not (page_text or "").strip():
            continue
        if doc_type in ("claim_note", "adjuster_note", "notes"):
            entries = _split_claim_notes(page_text)
            for entry in entries:
                for part in _window_split(entry, max_chars=max_chars, overlap=overlap):
                    idx += 1
                    chunk_id = f"clm_{claim_id}_{doc_id}_p{page_num}_{idx}"
                    md = dict(base_metadata)
                    md.update({"page": page_num, "section": "NOTE_ENTRY", "chunk_id": chunk_id, "doc_type": doc_type})
                    chunks.append(Chunk(chunk_id=chunk_id, text=part, metadata=md))
        else:
            blocks = _split_by_headings(page_text)
            for heading, body in blocks:
                for part in _window_split(body, max_chars=max_chars, overlap=overlap):
                    idx += 1
                    chunk_id = f"clm_{claim_id}_{doc_id}_p{page_num}_{idx}"
                    md = dict(base_metadata)
                    md.update({"page": page_num, "section": heading, "chunk_id": chunk_id, "doc_type": doc_type})
                    chunks.append(Chunk(chunk_id=chunk_id, text=part, metadata=md))
    return chunks
PY

In [ ]:
#Create embeddings.py

%%bash
set -e
cd "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant/app/rag"

# ---- app/rag/embeddings.py ----
cat > embeddings.py <<'PY'
from __future__ import annotations
from typing import List
from openai import OpenAI

DEFAULT_EMBED_MODEL = "text-embedding-3-small"

class Embedder:
    def __init__(self, model: str = DEFAULT_EMBED_MODEL):
        self.client = OpenAI()
        self.model = model

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        resp = self.client.embeddings.create(
            model=self.model,
            input=texts,
        )
        return [d.embedding for d in resp.data]
PY

echo "✅ Created app/rag/loaders.py and app/rag/embeddings.py"
ls -la app/rag

✅ Created app/rag/loaders.py and app/rag/embeddings.py
total 2
-rw------- 1 root root  492 Feb 23 03:29 embeddings.py
-rw------- 1 root root 1443 Mar 12 05:09 vectorstore.py


Create Common ingestion python code for both policies and claims (ingestion_common.py)

In [ ]:
%%bash
set -e

cd "/content/drive/My Drive/Agentic AI Projects/agentic_adjuster_assistant/app/rag"

# ---------- app/rag/ingest_common.py ----------
cat > ingest_common.py <<'PY'
from __future__ import annotations
from typing import Callable, Dict, Any, List, Tuple, Protocol, Optional
from openai import OpenAI

from app.rag.loaders import load_pdf_pages
from app.rag.vectorstore import ChromaStore

class ChunkLike(Protocol):
    chunk_id: str
    text: str
    metadata: Dict[str, Any]

ChunkerFn = Callable[[List[Tuple[int, str]], Dict[str, Any], int, int], List[ChunkLike]]

def ingest_pdf(
    *,
    pdf_path: str,
    base_metadata: Dict[str, Any],
    chunker: ChunkerFn,
    persist_dir: str,
    collection_name: str = "agentic_adjuster_assistant",
    embed_model: str = "text-embedding-3-small",
    max_chars: int = 1600,
    overlap: int = 200,
    reingest_filter: Optional[Dict[str, Any]] = None,
) -> int:
    """
    Generic ingestion:
      1) load PDF pages
      2) chunk with provided chunker(pages, base_metadata, max_chars, overlap)
      3) embed chunks
      4) upsert to Chroma

    reingest_filter:
      If provided, delete existing vectors that match this metadata filter
      before upserting (useful for doc_id+doc_version).
    """
    pages = load_pdf_pages(pdf_path)
    page_tuples = [(p.page, p.text) for p in pages]

    chunks = chunker(page_tuples, base_metadata, max_chars, overlap)
    if not chunks:
        return 0

    store = ChromaStore(persist_dir=persist_dir, collection_name=collection_name)
    if reingest_filter:
        store.delete_by_filter(reingest_filter)

    client = OpenAI()
    texts = [c.text for c in chunks]
    resp = client.embeddings.create(model=embed_model, input=texts)
    vectors = [d.embedding for d in resp.data]

    store.upsert(
        ids=[c.chunk_id for c in chunks],
        embeddings=vectors,
        documents=[c.text for c in chunks],
        metadatas=[c.metadata for c in chunks],
    )
    return len(chunks)
PY

# ---------- app/rag/ingest_policy.py ----------
cat > ingest_policy.py <<'PY'
from __future__ import annotations
from typing import Dict, Any, List, Tuple

from app.rag.chunking import chunk_policy_pages
from app.rag.ingest_common import ingest_pdf

def _policy_chunker(pages: List[Tuple[int, str]], base_md: Dict[str, Any], max_chars: int, overlap: int):
    return chunk_policy_pages(pages, base_metadata=base_md, max_chars=max_chars, overlap=overlap)

def ingest_policy_pdf(
    *,
    pdf_path: str,
    policy_metadata: Dict[str, Any],
    persist_dir: str,
    collection_name: str = "agentic_adjuster_assistant",
    embed_model: str = "text-embedding-3-small",
    max_chars: int = 1600,
    overlap: int = 200,
    reingest: bool = True,
) -> int:
    # For policies, reingest by doc_id + doc_version (if present)
    reingest_filter = None
    if reingest and policy_metadata.get("doc_id") and policy_metadata.get("doc_version"):
        reingest_filter = {"doc_id": policy_metadata["doc_id"], "doc_version": policy_metadata["doc_version"]}

    return ingest_pdf(
        pdf_path=pdf_path,
        base_metadata=policy_metadata,
        chunker=_policy_chunker,
        persist_dir=persist_dir,
        collection_name=collection_name,
        embed_model=embed_model,
        max_chars=max_chars,
        overlap=overlap,
        reingest_filter=reingest_filter,
    )
PY

echo "✅ Refactor complete: ingest_common.py + ingest_policy.py"

✅ Refactor complete: ingest_common.py + ingest_policy.py


In [ ]:
%%bash
set -e
# ---------- app/rag/ingest_claims.py ----------
cat > app/rag/ingest_claims.py <<'PY'
from __future__ import annotations
from typing import Dict, Any, List, Tuple

from app.rag.claims_chunking import chunk_claim_document
from app.rag.ingest_common import ingest_pdf

def _claims_chunker(pages: List[Tuple[int, str]], base_md: Dict[str, Any], max_chars: int, overlap: int):
    return chunk_claim_document(pages=pages, base_metadata=base_md, max_chars=max_chars, overlap=overlap)

def ingest_claim_pdf(
    *,
    pdf_path: str,
    claim_metadata: Dict[str, Any],
    persist_dir: str,
    collection_name: str = "agentic_adjuster_assistant",
    embed_model: str = "text-embedding-3-small",
    max_chars: int = 1800,
    overlap: int = 200,
    reingest: bool = True,
) -> int:
    required = ["claim_id", "policy_id", "doc_type", "doc_id", "doc_version"]
    missing = [k for k in required if not claim_metadata.get(k)]
    if missing:
        raise ValueError(f"Missing required claim_metadata keys: {missing}")

    # Reingest by doc_id + doc_version
    reingest_filter = {"doc_id": claim_metadata["doc_id"], "doc_version": claim_metadata["doc_version"]} if reingest else None

    return ingest_pdf(
        pdf_path=pdf_path,
        base_metadata=claim_metadata,
        chunker=_claims_chunker,
        persist_dir=persist_dir,
        collection_name=collection_name,
        embed_model=embed_model,
        max_chars=max_chars,
        overlap=overlap,
        reingest_filter=reingest_filter,
    )
PY
